# Install Necessary Packages

In [7]:
import os
import pandas as pd
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

## Code to read EPUB files into a plain text string for further text analysis

The EPUB files contain meta information about author, date, and title. While also conating OHCO information about Chapter and Paragraph these will all be extracted into the `books_df`.

All Books are on Box Cloud Storage

In [8]:
#File path for Epub files
epub_file_location = 'C:/Users/mason/Box/Text As Data Final/SandersonFiles'

In [9]:
#Read in the Epub files and extract text
def extract_epub_to_paragraphs(epub_path):
    book = epub.read_epub(epub_path)
    title_metadata = book.get_metadata('DC', 'title')
    title = title_metadata[0][0] if title_metadata else 'Unknown Title'
    author_metadata = book.get_metadata('DC', 'creator')
    author = author_metadata[0][0] if author_metadata else 'Unknown Author'
    date_metadata = book.get_metadata('DC', 'date')
    date = date_metadata[0][0] if date_metadata else 'Unknown Date'
    print(f"Getting: {title} info...")

    paragraph_data =  []

    documents = list(book.get_items_of_type(ebooklib.ITEM_DOCUMENT))

    for chap_id, item in enumerate(documents):
        soup = BeautifulSoup(item.get_content(), 'html.parser')
        paragraphs =soup.find_all('p')

        for para_id, para in enumerate(paragraphs):
            text = para.get_text(strip=True)

            if text and len(text) >1:
                paragraph_data.append({
                    'title': title,
                    'author': author,
                    'date': date,
                    'chapter_id': chap_id,
                    'paragraph_id': para_id,
                    'text': text

                })
    return paragraph_data

In [10]:
directory_path = epub_file_location
corpus_data = []

In [12]:
for filename in os.listdir(directory_path):
    if filename.endswith('.epub'):

        epub_path = os.path.join(directory_path, filename)

        try:
            book_paragraphs = extract_epub_to_paragraphs(epub_path)
            corpus_data.extend(book_paragraphs)
        except Exception as e:
            print(f"Failed to process {filename}. Error: {e}")

books_df = pd.DataFrame(corpus_data)
print("Books DataFrame created!")
print(books_df.head())


Getting: A Memory of Light info...
Getting: Arcanum Unbounded info...
Getting: Elantris info...
Getting: Infinity Blade: Awakening info...
Getting: Isles of the Emberdark info...
Getting: Oathbringer info...
Getting: Infinity Blade: Redemption info...
Getting: Rhythm of War (9781429952040) info...
Getting: Shadows of Self info...
Getting: Steelheart info...
Getting: The Aether of Night info...
Getting: The Alloy of Law: A Mistborn Novel info...
Getting: Sanderson, Brandon - Mistborn 06 - The Bands of Mourning info...
Getting: Mistborn: The Final Empire info...
Getting: The Hero of Ages info...
Getting: Lost Metal : A Mistborn Novel (9780765391209) info...
Getting: The Sunlit Man info...
Getting: Way of Kings info...
Getting: The Well of Ascension info...
Getting: The Wheel of Time info...
Getting: Unknown Title info...
Getting: Tress of the Emerald Sea info...
Getting: Warbreaker info...
Getting: Wind and Truth: The brand new epic Stormlight Archive novel from the international bestsel

## Save the Resulting Brandon Sanderson Corpus to Box

In [13]:
#Save the corpus to a CSV file
books_df.to_csv(f"{directory_path}/BrandonSandersonBooks.csv", index=False)
print("Books saved to CSV!")

Books saved to CSV!


## Break down the paragraphs into sentences and terms

The Dataframe (`books_df`) as it currently stands utilized the html like parsing of `'p'` to denote paragraphs so rather than throw those away they were added to the dataframe. One less OHCO step down the line.


In [14]:
import re

OHCO=['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id']

books_df['sentence'] = books_df['text'].str.split(r'(?<=[.!?])\s+')
df_sentences = books_df.explode('sentence').drop(columns=['text'])
df_sentences['sent_id'] = df_sentences.groupby(['title', 'chapter_id', 'paragraph_id']).cumcount()
df_sentences['token'] = df_sentences['sentence'].str.lower().str.findall(r'\b[\w\']+\b')
df_tokens = df_sentences.explode('token').drop(columns=['sentence'])
df_tokens = df_tokens.dropna(subset=['token'])
df_tokens['token_id'] = df_tokens.groupby(['title', 'chapter_id', 'paragraph_id', 'sent_id']).cumcount()
df_tokens = df_tokens.set_index(OHCO)

df_tokens.to_csv(f"{directory_path}/BrandonSandersonBooks_Tokens.csv")
print("Tokens saved to CSV!")

Tokens saved to CSV!


In [15]:
df_tokens.head()

author  \
title             chapter_id paragraph_id sent_id token_id                                      
A Memory of Light 0          0            0       0         Robert Jordan & Brandon Sanderson   
                                                  1         Robert Jordan & Brandon Sanderson   
                                                  2         Robert Jordan & Brandon Sanderson   
                             1            0       0         Robert Jordan & Brandon Sanderson   
                                                  1         Robert Jordan & Brandon Sanderson   

                                                                  date   token  
title             chapter_id paragraph_id sent_id token_id                      
A Memory of Light 0          0            0       0         2013-01-08      by  
                                                  1         2013-01-08  robert  
                                                  2         2013-01-08  jordan  
                             1            0       0         2013-01-08     the  
                                                  1         2013-01-08     eye

## Conclusion of this notebook

Okay so now we have a `books_df` and a `df_tokens`. Neither are in the `LIB`, `CORPUS`, or`VOCAB` format. The `books_df` dataframe contains too much OHCO and Text. The `df_tokens` doesn't contain POS and has metainformation about each work.